# Variational Autoencoders

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['axes.grid'] = True
plt.rcParams['axes.axisbelow'] = True
plt.rcParams['figure.dpi'] = 150

In [ ]:
# Check available hardware
import torch
if torch.cuda.is_available():
    print(f'GPU is available, using GPU {torch.cuda.get_device_name(0)}')
    device = torch.device('cuda')
elif torch.mps.is_available():
    print(f'Apple Metal Performance Shaders framework is available, using {torch.device("mps")}')
    device = torch.device('mps')
else:
    print(f'Hardware acceleration not available, using CPU')
    device = torch.device('cpu')

## Dataset: MNIST

In [ ]:
import torch
import torchvision
from torchvision.transforms import v2

transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True)
])
mnist = torchvision.datasets.MNIST(root='~/scr10/', train=True, transform=transform, download=True)
mnist_test = torchvision.datasets.MNIST(root='~/scr10/', train=False, transform=transform, download=False)
from torch.utils.data import random_split
mnist_train, mnist_val = random_split(mnist, [0.8, 0.2])
print('Training dataset has length', len(mnist_train))
print('Training dataset has length', len(mnist_val))
print('Training dataset has length', len(mnist_test))

In [ ]:
x, y = mnist_train[0]
print('Input has shape', x.shape, 'and the output is', y)

In [ ]:
fig, ax = plt.subplots(2, 5, sharex=True, sharey=True)
ax = ax.flatten()
for ii in range(10):
    x, y = mnist_train[np.random.choice(len(mnist_train))]
    ax[ii].imshow(x.squeeze(), cmap='Greys')
    ax[ii].set(xticks=[], yticks=[])
plt.tight_layout()

## Variational autoencoder

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        self.fc = nn.Linear(64*3*3, 4)

    def forward(self, x):
        # [B, 1, 28, 28]
        x = self.conv1(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)

        # [B, 16, 14, 14]
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)

        # [B, 32, 7, 7]
        x = self.conv3(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)

        # [B, 64, 3, 3]
        x = x.reshape([-1, 64*3*3])
        x = self.fc(x)

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.ConvTranspose2d(16, 1, kernel_size=3, padding=1)
        self.conv2 = nn.ConvTranspose2d(32, 16, kernel_size=3, padding=1)
        self.conv3 = nn.ConvTranspose2d(64, 32, kernel_size=3, padding=1)

        self.fc = nn.Linear(4, 64*3*3)

    def forward(self, x):
        # [B, 10]
        x = self.fc(x)

        # [B, 64*3*3]
        x = x.reshape([-1, 64, 3, 3])

        x = F.interpolate(x, size=(7, 7))
        x = self.conv3(x)
        x = F.relu(x)

        x = F.interpolate(x, size=(14, 14))
        x = self.conv2(x)
        x = F.relu(x)

        x = F.interpolate(x, size=(28, 28))
        x = self.conv1(x)
        x = F.relu(x)

        return x

class VariationalAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def forward(self, x):

model = VariationalAutoencoder()

In [ ]:
from tqdm.notebook import tqdm, trange
from torch.utils.data import DataLoader

class VAELoss(nn.Module):

def train(model, loss_fn, optimizer, train_dataset, val_dataset, 
          batch_size=32, num_epochs=50, num_workers=2, device=device):
    # Send model to correct hardware device
    model = model.to(device)

    # Create dataloaders for train/validation
    train_loader = DataLoader(train_dataset, batch_size=batch_size, num_workers=num_workers, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, num_workers=num_workers, shuffle=False)

    logging = {
    }
    
    for epoch in range(num_epochs):
        for key in logging:
            logging[key].append(0.)
        
        model.train()
        for x, _, in tqdm(train_loader, leave=False):
            # Send x, y to GPU/TPU/etc.
            x = x.to(device)

            # Make a prediction, compute the loss

            # Backpropagate gradients, apply weight updates, reset gradients
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            # Logging
            for key, value in logs.items():
                logging['train_' + key][-1] += value * x.shape[0]

        model.eval()
        with torch.no_grad():
            for x, _, in tqdm(val_loader, leave=False):
                # Send x, y to GPU/TPU/etc.
                x = x.to(device)
    
                # Make a prediction, compute the loss
    
                # Logging
                for key, value in logs.items():
                    logging['val_' + key][-1] += value * x.shape[0]

        outstr = f'Epoch {epoch+1} |'
        for key in logging.keys():
            div = len(train_dataset) if key.startswith('train') else len(val_dataset)
            logging[key][-1] = logging[key][-1] / div
            outstr += f' {key} {logging[key][-1]:.3f} |'
        print(outstr)

    return model, logging

model = VariationalAutoencoder()
loss = VAELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
model, model_stats = train(
    model, 
    loss, 
    optimizer, 
    mnist_train, 
    mnist_val, 
    num_epochs=10,
    batch_size=64,
    num_workers=4,
)

In [ ]:
fig, ax = plt.subplots(1, 3, sharex=True, figsize=(7.5, 2.5))

for i, key in enumerate(['loss', 'recon', 'kld']):
    ax[i].plot(model_stats['train_'+key], color='black', label='Train')
    ax[i].plot(model_stats['val_'+key], color='red', label='Validation')
    ax[i].set(xlabel='Epoch')

ax[0].set_title('Loss')
ax[1].set_title(r'$\log q_{\phi}(x)$')
ax[2].set_title(r'$D_{KL} [ p_{\theta}(z | x) \, \Vert \, q(z) ]$')

fig.legend(*ax[0].get_legend_handles_labels(), 
           loc='center left', bbox_to_anchor=[1, 0.5], framealpha=0)

plt.tight_layout()

### Inference

In [ ]:
fig, ax = plt.subplots(2, 5, figsize=(10, 4), sharex=True, sharey=True)
for ii in range(5):
    xb, _ = mnist_test[ii]
    ax[0, ii].imshow(xb.squeeze().cpu().numpy(), cmap='Greys')
    with torch.no_grad():
        x_pred, z_pred, params = model(xb.to(device))
    ax[1, ii].imshow(x_pred.squeeze().cpu().numpy(), cmap='Greys')

ax[0,0].set_ylabel('Original')
ax[1,0].set_ylabel('Reconstructed')

for a in ax.flatten():
    a.set(xticks=[], yticks=[])
plt.tight_layout()

In [ ]:
# Inference
model.eval()
test_loader = DataLoader(mnist_test, batch_size=128, shuffle=False)

y_true = []
z_pred = []

with torch.no_grad():
    for xb, yb in tqdm(test_loader,leave=False):
        xb = xb.to(device)
        xp, zp, params = model(xb)
        y_true.append(yb.cpu().numpy())
        z_pred.append(zp.cpu().numpy())

y_true = np.concatenate(y_true)
z_pred = np.concatenate(z_pred)

In [ ]:
import seaborn as sns

df = pd.DataFrame({
    'z0': z_pred[:, 0],
    'z1': z_pred[:, 1],
    'z2': z_pred[:, 2],
    'z3': z_pred[:, 3],
    'y': y_true
})

sns.pairplot(df, hue='y', palette='tab10', plot_kws={'s': 5})

In [ ]:
from sklearn.manifold import TSNE

viz = TSNE(n_components=2, n_iter=250)
z_2d = viz.fit_transform(z_pred)

fig, ax = plt.subplots()
ax.scatter(
    z_2d[:, 0],
    z_2d[:, 1],
    c=y_true,
    cmap='tab10',
    s=5
)
ax.set(xlabel='t-SNE 1', ylabel='t-SNE 2')

## $\beta$-VAE

In [ ]:
class BetaVAELoss(nn.Module):
    def __init__(self, beta=1.):
        super().__init__()
        self.beta = beta
        
    def forward(self, x_pred, x_true, mu, logvar):
        

# Fix training recipe
train_kwargs = dict(train_dataset=mnist_train, val_dataset=mnist_val, num_epochs=10, batch_size=64, num_workers=4)

In [ ]:
# Model with heavy regularization
model1 = VariationalAutoencoder()
opt1 = torch.optim.Adam(model1.parameters(), lr=1e-4)
loss1 = BetaVAELoss(beta=10.)

model1, model1_stats = train(model1, loss1, opt1, **train_kwargs)

In [ ]:
# Model with minimal regularization
model2 = VariationalAutoencoder()
opt2 = torch.optim.Adam(model2.parameters(), lr=1e-4)
loss2 = BetaVAELoss(beta=1e-2)

model2, model2_stats = train(model2, loss2, opt2, **train_kwargs)

In [ ]:
fig, ax = plt.subplots(1, 3, sharex=True, figsize=(7.5, 2.5))

for i, key in enumerate(['loss', 'recon', 'kld']):
    ax[i].plot(model_stats['val_'+key], label='$\\beta = 1$')
    ax[i].plot(model1_stats['val_'+key], label='$\\beta = 10$')
    ax[i].plot(model2_stats['val_'+key], label='$\\beta = 0.01$')
    ax[i].set(xlabel='Epoch')

ax[0].set_title('Loss')
ax[1].set_title(r'$\log q_{\phi}(x)$')
ax[2].set_title(r'$D_{KL} [ p_{\theta}(z | x) \, \Vert \, q(z) ]$')

fig.legend(*ax[0].get_legend_handles_labels(), 
           loc='center left', bbox_to_anchor=[1, 0.5], framealpha=0)

plt.tight_layout()

### Inference

In [ ]:
fig, ax = plt.subplots(4, 5, figsize=(10, 8), sharex=True, sharey=True)
for ii in range(5):
    xb, _ = mnist_test[ii]
    ax[0, ii].imshow(xb.squeeze().cpu().numpy(), cmap='Greys')
    with torch.no_grad():
        x_pred, z_pred, params = model(xb.to(device))
        ax[1, ii].imshow(x_pred.squeeze().cpu().numpy(), cmap='Greys')

        x_pred, z_pred, params = model1(xb.to(device))
        ax[2, ii].imshow(x_pred.squeeze().cpu().numpy(), cmap='Greys')

        x_pred, z_pred, params = model2(xb.to(device))
        ax[3, ii].imshow(x_pred.squeeze().cpu().numpy(), cmap='Greys')

ax[0,0].set_ylabel('Original')
ax[1,0].set_ylabel('$\\beta = 1$')
ax[2,0].set_ylabel('$\\beta = 10$')
ax[3,0].set_ylabel('$\\beta = 0.01$')

for a in ax.flatten():
    a.set(xticks=[], yticks=[])
plt.tight_layout()

In [ ]:
# Inference
model.eval()
test_loader = DataLoader(mnist_test, batch_size=128, shuffle=False)

y_true = []
z0_pred = []
z1_pred = []
z2_pred = []
with torch.no_grad():
    for xb, yb in tqdm(test_loader,leave=False):
        xb = xb.to(device)
        y_true.append(yb.cpu().numpy())

        xp, zp, params = model(xb)
        z0_pred.append(zp.cpu().numpy())

        xp, zp, params = model1(xb)
        z1_pred.append(zp.cpu().numpy())

        xp, zp, params = model2(xb)
        z2_pred.append(zp.cpu().numpy())

y_true = np.concatenate(y_true)
z0_pred = np.concatenate(z0_pred)
z1_pred = np.concatenate(z1_pred)
z2_pred = np.concatenate(z2_pred)

In [ ]:
from sklearn.manifold import TSNE

viz = TSNE(n_components=2, n_iter=250)
z0_2d = viz.fit_transform(z0_pred)
z1_2d = viz.fit_transform(z1_pred)
z2_2d = viz.fit_transform(z2_pred)

In [ ]:
from sklearn.manifold import TSNE

fig, axes = plt.subplots(3, 1, figsize=(5, 15))

for ax, z_2d in zip(axes, [z0_2d, z1_2d, z2_2d]):
    ax.scatter(
        z_2d[:, 0],
        z_2d[:, 1],
        c=y_true,
        cmap='tab10',
        s=5
    )
    ax.set(xlabel='t-SNE 1', ylabel='t-SNE 2')

axes[0].set_title('$\\beta = 1$')
axes[1].set_title('$\\beta = 10$')
axes[2].set_title('$\\beta = 0.01$');

## Latent traversal

In [ ]:
fig, ax = plt.subplots(4, 8, figsize=(8, 4), sharex=True, sharey=True)

with torch.no_grad():
    z_base = torch.randn(1, 4, device=device, dtype=torch.float) # Starting latent vector
    z_trav = torch.linspace(-3, 3, 8, device=device, dtype=torch.float) # Traversal values

    for ii in range(4):
        z = z_base.repeat(8, 1)
        z[:, ii] = z_trav
        x_pred = model.decoder(z).cpu().numpy()
        for jj in range(8):
            ax[ii, jj].imshow(x_pred[jj,0], cmap='Greys')
            ax[ii, jj].set(xticks=[], yticks=[])